# Error Metrics

When assessing the accuracy of a SINDy model, we follow the metrics used by [(Messenger and Bortz, 2021)](https://epubs.siam.org/doi/abs/10.1137/20M1343166):
1. The relative Frobenius norm error
2. True positive ratio

We also employ coefficient of determination ($R^2$ error) metric which is the default metric used in PySINDy for evaluating model prediction and simulation.

These metrics are used to evaluate different aspects of the model:
1. The accuracy of the model's coefficient by looking at the element-wise difference between the produced and expected $\Xi$ matrix (relative Frobenius norm)
2. The sparsity levels of the produced coefficient matrix $\Xi$ compared to the sparsity of the true $\Xi$ matrix (true positive ratio)
3. The difference in the predicted time derivative ($R^2$ error)
4. The difference in the resulting trajectory ($R^2$ error)

In [ ]:
from sindy.metrics import rel_fro_err, precision
from sklearn.metrics import r2_score

# Relative Frobenius Norm Error

Let us define the relative Frobenius norm error as

$$E(\Xi, \Xi^\star) = \frac{||\Xi - \Xi^\star||_F}{||\Xi^\star||_F},$$

where $\Xi^\star$ is the true coefficient matrix. This may be directly applied to evaluate the accuracy of the coefficients learned by SINDy.

Additionally, this formula may also be used to evaluate the accuracy of the predicted time derivative given by

$$\dot{X} = \Theta(X) \Xi,$$

or the resulting trajectory by applying an integrator on the learned SINDy model. An implementation of this error term is found in `rel_fro_err`.

In [2]:
rel_fro_err??

Signature: rel_fro_err(eval_mat: numpy.ndarray, true_mat: numpy.ndarray) -> float
Source:   
def rel_fro_err(
        eval_mat: np.ndarray,
        true_mat: np.ndarray
) -> float:
    """
    Calculates the relative Frobenius norm error between two matrices, which
    provides a measure for the element wise error between two matrices.

    The relative Frobenius norm error is defined as the Frobenius norm of the
    difference between `eval_mat` and `true_mat`, divided by the Frobenius
    norm of `true_mat`. This metric quantifies the overall difference between
    the two matrices, normalized by the magnitude of the true matrix.

    Args:
        eval_mat (np.ndarray): The evaluated matrix.
        true_mat (np.ndarray): The true matrix.

    Returns:
        float: The relative Frobenius norm error. A value of 0 indicates
            perfect agreement between the two matrices.
    """
    
    return np.linalg.norm(eval_mat - true_mat, ord='fro') / \
        np.linalg.norm(true_ma

# Precision

The precision is defined as

$$\text{TPR} = \frac{\text{correct positive classifications}}{\text{all positive classifications}} = \frac{\text{TP}}{\text{TP} + \text{FP}},$$

where TP is the true positive and FP is the false negative. It is implemented in `precision`.

In [ ]:
precision??

Signature: true_positive_ratio(eval_mat: numpy.ndarray, true_mat: numpy.ndarray) -> float
Source:   
def true_positive_ratio(
        eval_mat: np.ndarray,
        true_mat: np.ndarray
) -> float:
    """
    Calculates the true positive ratio between two matrices which provides a
    measure of how many correct non-zero terms were identified.

    It is defined by the ratio between true positives, and the sum of true
    positives, false negatives, and false positives.

    Args:
        eval_mat (np.ndarray): The evaluated matrix, where non-zero elements
            indicate selected features or active terms.
        true_mat (np.ndarray): The true matrix, where non-zero elements
            indicate the actual selected features or active terms.

    Returns:
        float: The true positive ratio. A value of 1 indicates perfect
            recovery of the sparsity pattern. A value of 0 indicates no
            overlap in the sparsity patterns.
    """
    
    eval_mask: np.ndarray 

# Coefficient of Determination

The coefficient of determination (or the $R^2$ error) is a statistical measure of the fit of a model's output, whilst also considering its ability to predict the existing variance within the dataset. It is defined by

$$R^2(\boldsymbol{y}, \boldsymbol{y}^\star) = 1 - \frac{\sum_i \left( \boldsymbol{y}^\star_i - \boldsymbol{y}_i \right)^2}{\sum_i \left( \boldsymbol{y}^\star_i - \bar{\boldsymbol{y}} \right)^2},$$

where $\boldsymbol{y}$ is the predicted data, $\boldsymbol{y}^\star$ is the actual data, and $\bar{\boldsymbol{y}}$ is the mean of the actual data. A perfect score of 1 indicates that the the prediction matches the variance from the data. PySINDy uses the `r2_score` implementation from Scikit-learn.

In [ ]:
r2_score??

Signature:
r2_score(
    y_true,
    y_pred,
    *,
    sample_weight=None,
    multioutput='uniform_average',
    force_finite=True,
)
Source:   
@validate_params(
    {
        "y_true": ["array-like"],
        "y_pred": ["array-like"],
        "sample_weight": ["array-like", None],
        "multioutput": [
            StrOptions({"raw_values", "uniform_average", "variance_weighted"}),
            "array-like",
            None,
        ],
        "force_finite": ["boolean"],
    },
    prefer_skip_nested_validation=True,
)
def r2_score(
    y_true,
    y_pred,
    *,
    sample_weight=None,
    multioutput="uniform_average",
    force_finite=True,
):
    """:math:`R^2` (coefficient of determination) regression score function.

    Best possible score is 1.0 and it can be negative (because the
    model can be arbitrarily worse). In the general case when the true y is
    non-constant, a constant model that always predicts the average y
    disregarding the input features would get a